## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import gc
from xgboost import XGBRanker

from Challenge.paths import generate_submission, load_xgboost_cv_folds, XGBOOST_DIR, XGBOOST_DATAFRAMES
from Challenge.utils import evaluate_recommender
from Challenge.XGBoostReranker import XGBoostRerankerRecommender

Running on local — storage at: /home/luigi/RecSys


## **Load XG Boost Model**

In [11]:
model_path = os.path.join(XGBOOST_DIR, "xgb_meta_ranker_baseline.joblib")

xgb_model = joblib.load(model_path)

print(f"✅ XGBRanker model successfully loaded from disk.")

✅ XGBRanker model successfully loaded from disk.


## **Load Data**

In [12]:
file_path = os.path.join(XGBOOST_DATAFRAMES, "validation_data.parquet")

X_pred = pd.read_parquet(
    file_path,
    engine='fastparquet'
)

print(f"Loaded successfully from: {file_path}")
X_pred

Loaded successfully from: /home/luigi/RecSys/xg_boost_data/dataframes/validation_data.parquet


,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,2530,False,0.024645,1879,0,0.481208,112,0,0.401178,...,380.181824,739.703247,2.558068,6.539532,0.403210,0.286240,0.547355,-0.601053,88,299
1,0,2392,False,0.029240,1686,0,0.521009,77,0,0.468675,...,357.545441,671.232422,2.176412,3.457409,0.336442,0.217113,0.773426,0.954189,88,368
2,0,1968,True,0.148956,326,0,0.529162,69,0,0.470274,...,249.500000,813.374756,4.599273,21.388342,0.340657,0.238187,0.935191,1.076236,88,1794
3,0,1252,False,0.478947,20,0,0.703392,15,0,0.749076,...,580.454529,1369.205811,2.454054,5.036523,0.456903,0.272891,-0.242999,-0.588007,88,5719
4,0,3142,False,0.108688,508,0,0.370004,235,0,0.268305,...,369.363647,334.381805,1.659096,2.413729,0.207367,0.209059,2.452234,7.687963,88,1316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4062128,27094,4180,False,0.372348,46,0,0.037867,2133,0,0.000000,...,3401.272705,2429.000000,0.145498,-1.646870,0.052230,0.247073,2.366482,8.813674,285,4425
4062129,27094,5167,False,0.357895,47,0,0.056512,1709,0,0.000000,...,3578.045410,1970.676514,-0.580727,-0.788151,0.035437,0.252919,2.312291,9.667974,285,4309
4062130,27094,531,False,0.514286,16,0,0.065830,1540,0,0.000000,...,2737.500000,2082.018311,0.650098,-0.615000,0.075414,0.262491,1.887378,6.194579,285,6122
4062131,27094,3103,False,0.049875,1080,0,0.212755,452,0,0.054115,...,745.772705,1173.984497,3.481864,13.444949,0.198115,0.246998,1.439667,3.249488,285,603


In [13]:
recommender = XGBoostRerankerRecommender(xgb_model, X_pred)

In [14]:
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

_, URM_test_complete = load_holdout_split()
_, URM_validation = split_train_in_two_percentage_global_sample(URM_test_complete, train_percentage = 0.5)

In [15]:
evaluate_recommender(recommender, URM_validation=URM_validation, at=20)

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 12.65it/s]


np.float64(0.24487366327348611)